# DAS fin-whale localization — Deep EDA
### Get to know the data: a fiber-optic cable as a 46,000-sensor array

**Task 3 of DCLDE 2027** — detect and **localize** fin-whale calls on Distributed Acoustic Sensing (DAS):
two fiber cables (North, South) off Oregon, each turned into tens of thousands of strain sensors. The data is
a **strain image** (channel × time), and detection/localization is effectively **computer vision**, not
per-hydrophone PAM.

> **Read this first — the ground truth is *arrival picks*, not positions.** Each of the 11,117 rows is one
> call observed at one channel (`time`, `dist` along cable, `call_type` LF/HF, `snr`); rows sharing
> `(file, segment_id, call_id)` trace a call's **moveout curve** (1,221 distinct calls). There is **no
> lat/lon/range label** — the source location is implicit in the moveout, and **97 % of calls capture only
> one limb** of the V, so single-cable range is under-constrained.

*Loads a committed calls snapshot; fetches cable geometry + one 0.5 GB `.nc` strain segment. `DCLDE 2027 (uv)` kernel.*

## What is this data? (plain-language primer — terms defined before we use them)

**The animal.** **Fin whale** (*Balaenoptera physalus*) — the second-largest animal on Earth. It sings very low, repetitive **~20 Hz pulses** (below human hearing).

**The sensor — DAS.** **DAS = Distributed Acoustic Sensing.** Instead of microphones, a laser is sent down an ordinary **fiber-optic cable** lying on the seafloor; tiny stretches of the glass (**strain**) caused by passing sound are read out **every ~2 m** along the cable. One cable becomes tens of thousands of vibration sensors (**channels**). Here two cables — **North** and **South** — run off the Oregon coast.

**The data — a "strain image".** Each 70-second window is a 2-D array: **channel (distance along the cable) × time**, stored as **NetCDF (`.nc`)** files. Values are denoised **SNR (Signal-to-Noise Ratio)** in two frequency bands: **LF** (low-frequency = the ~20 Hz pulse) and **HF** (a higher-frequency component). So detection here is really **image analysis**, not listening.

**Other terms used below:** **channel** = one sensing point on the cable, its position given as **distance-along-cable** (0–95 km); **moveout curve** = because sound reaches near channels sooner than far ones, one call draws a **V / chevron** in the channel×time image; **CPA (Closest Point of Approach)** = where the whale is nearest the cable, the **apex** of that V; **gauge length (GL)** = the length of fiber averaged per reading (~51 m); **fs** = sampling rate (200 Hz → 100 Hz Nyquist); **localization** = inferring the whale's position from the moveout picks.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, REPO)
os.environ.setdefault("DCLDE_SAMPLE_DIR",
    "/opt/dlami/nvme/dclde_samples" if os.path.isdir("/opt/dlami/nvme") else os.path.join(REPO, "data", "_samples"))
os.environ.setdefault("MPLCONFIGDIR", "/opt/dlami/nvme/mpl" if os.path.isdir("/opt/dlami/nvme") else "/tmp/mpl")
from scripts.fetch_sample import fetch
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="talk")

df = pd.read_csv(os.path.join(os.getcwd(), "das_calls.csv.gz"))   # 11,117 annotated call-picks
df["utc_time"] = pd.to_datetime(df["utc_time"]); df["dist_km"] = df["dist"] / 1000
df["hour"] = df["utc_time"].dt.hour
geo = {c.capitalize(): pd.read_csv(fetch(f"dclde_2027_das-finwhale-localization/geometry/{c}_DAS_multicoord.csv")).sort_values("chan_m")
       for c in ("north", "south")}
def to_geo(sub, cable):
    g = geo[cable]; idx = np.clip(np.searchsorted(g.chan_m.values, sub.dist.values), 0, len(g)-1)
    return g.lat.values[idx], g.lon.values[idx], g.depth.values[idx]
# per-call tracks (a row = one call at one channel; a track = one call across channels)
TRACK = ["src", "segment_id", "call_id"]
print(f"{len(df):,} call-picks | {df.groupby(TRACK).ngroups:,} distinct calls (tracks) | "
      f"cables {dict(df.cable.value_counts())} | types {dict(df.call_type.value_counts())}")

## 1. Coverage — 5 diel windows, two cables, LF & HF
Not continuous: 5 batches are 5 diel snapshots (~140 min total strain). South carries ~2/3 of calls; HF
outnumbers LF. Every cable×type cell is populated.

In [ ]:
g = df.groupby(["batch","cable","call_type"]).size().reset_index(name="n")
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
sns.barplot(data=g, x="batch", y="n", hue="cable", ax=ax[0], errorbar=None, estimator=sum)
ax[0].set(title="Call-picks per batch (5 diel windows)", xlabel="batch", ylabel="picks")
ct = df.groupby(["cable","call_type"]).size().unstack()
ct.plot(kind="bar", ax=ax[1], color={"HF":"#35a7c2","LF":"#e08a2b"}, width=.75)
ax[1].set(title="Cable x call-type (HF/LF)", xlabel="", ylabel="picks"); ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

## 2. Where along the cable the whales are (+ bathymetry)
Calls are strongly **non-uniform**: none in the first ~12.5 km (nearshore), clustering toward the deep far
ends. South has a hard empty band ~28–33 km — a **dead/masked cable section (instrument artifact, not
biology)**. Calls track deeper water / the shelf break.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5), sharey=False)
for a, cab in zip(ax, ["North","South"]):
    s = df[df.cable == cab]
    a.hist([s[s.call_type=="LF"].dist_km, s[s.call_type=="HF"].dist_km], bins=30, stacked=True,
           label=["LF","HF"], color=["#e08a2b","#35a7c2"])
    a2 = a.twinx()
    g = geo[cab]; a2.plot(g.chan_m/1000, g.depth, color="k", lw=1.2, alpha=.6)
    a2.set_ylabel("seafloor depth (m)"); a2.invert_yaxis()
    a.set(title=f"{cab} cable — calls vs distance", xlabel="distance along cable (km)", ylabel="picks"); a.legend(fontsize=10)
plt.tight_layout(); plt.show()
print("No calls in the first ~12.5 km (nearshore); South has a dead band ~28-33 km (instrument, not biology).")

## 3. SNR — and detectability falls with range
SNR (0–31, mean ~14) is similar for LF/HF; North averages slightly higher. SNR weakly anti-correlates with
distance-along-cable — far calls are fainter, as expected — which bounds where labels are trustworthy.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.violinplot(data=df, x="cable", y="snr", hue="call_type", split=True, ax=ax[0],
               palette={"HF":"#35a7c2","LF":"#e08a2b"}, cut=0)
ax[0].set(title="SNR by cable x call-type", xlabel="", ylabel="SNR")
ax[1].hexbin(df.dist_km, df.snr, gridsize=30, cmap="magma", mincnt=1)
ax[1].set(title="SNR vs distance (far = fainter)", xlabel="distance along cable (km)", ylabel="SNR")
plt.tight_layout(); plt.show()

## 4. The fin-whale song rhythm is in the labels
Interval between successive calls within a concurrent-source group shows a sharp **20–22 s** peak — the
classic fin-whale 20-Hz song inter-note interval. The broad 0–8 s mode is multiple overlapping whales (up to
4 per segment) interleaving. A real biological signal, straight from the annotations.

In [ ]:
# inter-call interval within a concurrent-source group -> fin-whale 20-s song rhythm
iv = []
for _, grp in df.sort_values("time").groupby(["src","segment_id"]):
    iv += np.diff(np.sort(grp.time.values)).tolist()
iv = np.array([v for v in iv if 0 < v < 60])
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(iv, bins=60, ax=ax, color="#7a4fb5")
ax.axvspan(20, 22, color="#e08a2b", alpha=.25); ax.annotate("20-22 s\n= fin-whale\nsong interval", (22.5, ax.get_ylim()[1]*0.6), fontsize=11)
ax.set(title="Inter-call interval (the song rhythm is in the labels)", xlabel="seconds", ylabel="intervals")
plt.tight_layout(); plt.show()

## 5. Moveout curves & the physics check
Each call sweeps across channels as a V; the **apex = closest point of approach**. The apparent along-cable
velocity has a sharp floor at **~1490 m/s = the sound speed of seawater** — a clean validity check (moveout
can't beat sound in water). Most tracks show a single near-linear limb (see next).

In [ ]:
# reconstruct per-call moveout (dist vs time) and apparent along-cable velocity
tracks = df.groupby(TRACK)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
n = 0
for _, t in tracks:
    if len(t) >= 6 and n < 60:
        t = t.sort_values("time"); ax[0].plot(t.time, t.dist_km, lw=.8, alpha=.5,
            color=("#35a7c2" if t.cable.iloc[0]=="North" else "#e08a2b")); n += 1
ax[0].set(title="Moveout curves (dist vs time) — apex = closest approach", xlabel="time in segment (s)", ylabel="distance (km)")
vel = []
for _, t in tracks:
    if len(t) >= 4:
        t = t.sort_values("time").values; d = np.diff([r[df.columns.get_loc("dist")] for r in t]); dt = np.diff([r[df.columns.get_loc("time")] for r in t])
        v = np.abs(d/np.where(dt==0, np.nan, dt)); vel += v[np.isfinite(v)].tolist()
vel = np.array([v for v in vel if 0 < v < 6000])
sns.histplot(vel, bins=60, ax=ax[1], color="#3b7dd8")
ax[1].axvline(1490, color="k", ls="--"); ax[1].annotate("1490 m/s\n= sound in water\n(hard floor)", (1550, ax[1].get_ylim()[1]*0.6), fontsize=11)
ax[1].set(title="Apparent along-cable velocity", xlabel="m/s", ylabel="segments"); ax[1].set_xlim(0, 6000)
plt.tight_layout(); plt.show()

## 6. Ground truth #1 — what the labels actually are (and their limits)
The GT is a set of **arrival picks**, and **only ~2.4 % of calls capture the apex** (both limbs). For the
other 97 %, the closest-approach point lies off the end of the observed channel span, so **single-cable
range is fundamentally under-constrained** — a distant broadside whale and a nearer end-fire whale give the
same limb. Well-posed localization needs the rare apex track, or **two-cable triangulation** (whose
cross-cable association is *not* provided). Median 8 picks per call.

In [ ]:
# GT reality: each track is arrival PICKS, and 97% capture only one limb of the V (apex off-span)
rows = []
for key, t in df.groupby(TRACK):
    t = t.sort_values("time"); dpos = t.dist.values
    amin = int(np.argmin(dpos)); interior = 0 < amin < len(t)-1     # apex observed inside the span?
    rows.append({"cable": t.cable.iloc[0], "n": len(t), "apex": interior})
tk = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(tk.n.clip(upper=25), bins=24, ax=ax[0], color="#7a4fb5")
ax[0].set(title=f"Points per call (median {int(tk.n.median())})", xlabel="channel-picks per call", ylabel="calls")
share = tk.apex.mean()
ax[1].bar(["single-limb\n(apex off-span)","apex captured\n(V, well-posed)"], [1-share, share], color=["#c76b3a","#3b7dd8"])
ax[1].set(title=f"Only {share:.1%} of calls capture the apex", ylabel="fraction of calls")
for i, v in enumerate([1-share, share]): ax[1].annotate(f"{v:.1%}", (i, v), ha="center", va="bottom", fontsize=12)
plt.tight_layout(); plt.show()
print("=> single cable + single limb => per-call range is under-constrained; the well-posed calls are the near-cable ones. Two-cable triangulation needed, but the cross-cable association is NOT given.")

## 7. Ground truth #2 — reconstructed source locations
Mapping each call's closest-approach channel to real lat/lon/depth via the geometry gives the implied
**"where are the whales" map**: they tile both cables over the continental slope (median seafloor ~500 m;
North shallower, South out to ~1500 m). This is the localization target's real-world footprint.

In [ ]:
# reconstructed source-location GT: map each call's closest-approach channel to real lat/lon/depth
cpa = []
for key, t in df.groupby(TRACK):
    r = t.loc[t.time.idxmin()]                      # earliest arrival ~ closest-point-of-approach proxy
    lat, lon, dep = to_geo(pd.DataFrame([r]), r.cable)
    cpa.append({"cable": r.cable, "lat": lat[0], "lon": lon[0], "depth": dep[0]})
cpa = pd.DataFrame(cpa)
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
for cab, col in [("North","#35a7c2"),("South","#e08a2b")]:
    g = geo[cab]; ax[0].plot(g.lon, g.lat, color=col, lw=1, alpha=.4)
sc = ax[0].scatter(cpa.lon, cpa.lat, c=cpa.depth, s=14, cmap="viridis_r")
plt.colorbar(sc, ax=ax[0], label="seafloor depth (m)")
ax[0].set(title="Implied whale locations (CPA) over the Oregon slope", xlabel="lon", ylabel="lat")
sns.violinplot(data=cpa, x="cable", y="depth", ax=ax[1], cut=0, palette={"North":"#35a7c2","South":"#e08a2b"})
ax[1].invert_yaxis(); ax[1].set(title="Seafloor depth at closest approach", xlabel="", ylabel="depth (m)")
plt.tight_layout(); plt.show()

## 8. Call distributions — length & band
The `.nc` is a **band-split denoised-SNR envelope** (`SNR_lf`/`SNR_hf`, 200 Hz → 100 Hz Nyquist), not raw
strain, so exact carrier frequencies aren't recoverable — but `call_type` is objectively the band carrying
the energy. LF = ~20-Hz pulse; HF = the higher component. Per-note duration ~0.4 s; a call's *footprint* in
the image spans ~5 s × ~9 km (moveout), which is what a detector must segment.

In [ ]:
# call "size" in the strain image + LF/HF split (the .nc is a band-split SNR envelope, no raw waveform)
span = df.groupby(TRACK).agg(tspan=("time", lambda s: s.max()-s.min()),
                             dspan=("dist", lambda s: (s.max()-s.min())/1000), cable=("cable","first"))
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
sns.histplot(span.tspan.clip(upper=30), bins=40, ax=ax[0], color="#5aa469")
ax[0].set(title=f"Call footprint in time (median {span.tspan.median():.1f} s)", xlabel="track time-span (s)", ylabel="calls")
sns.histplot(span.dspan.clip(upper=40), bins=40, ax=ax[1], color="#3b7dd8")
ax[1].set(title=f"Call footprint along cable (median {span.dspan.median():.1f} km)", xlabel="along-cable span (km)", ylabel="calls")
plt.tight_layout(); plt.show()
print("LF = ~20-Hz fin-whale pulse band; HF = higher-frequency component. call_type = the band carrying the energy. Per-note duration ~0.4 s.")

## 9. The strain image — the computer-vision view
Finally, the raw material: one 70 s × 64 km strain image. Whale calls are **luminous chevron ridges** sweeping
tens of km of fiber; the cyan picks are the GT sitting on the ridges. Detection/localization = finding and
fitting these curves. Full res is 185 M pixels/band with ~2.5 % positive — an extreme sparse-signal CV problem
(~4 TB/day/band at full acquisition).

In [ ]:
import xarray as xr
nc = fetch("dclde_2027_das-finwhale-localization/detections/batch_1/Denoised_SNR_North_2021-11-04_02-05-52.nc")
ds = xr.open_dataset(nc)
calls = pd.read_csv(fetch("dclde_2027_das-finwhale-localization/annotations/batch1/annotated_calls_north_2021-11-04_02_05_52.csv"))
t = ds["time"].values; d = ds["dist"].values/1000
sd = max(1, ds.sizes["dist"]//1500); st = max(1, ds.sizes["time"]//1500)
img = ds["SNR_lf"].values[::sd, ::st]
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.imshow(img, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=np.percentile(img, 99.5), extent=[t[0], t[-1], d[0], d[-1]])
sub = calls[calls.call_type=="LF"]
ax.scatter(sub.time, sub.dist/1000, s=16, facecolors="none", edgecolors="cyan", lw=1)
ax.set(title="Strain image (SNR_lf, 70 s x 64 km) — calls trace luminous moveout chevrons", xlabel="time (s)", ylabel="distance along cable (km)")
plt.tight_layout(); plt.show()

## Pitfalls — keep these on a slide
1. **GT = arrival picks, not positions** — you infer location; **97 % single-limb** → single-cable range under-constrained.
2. **Cross-cable association is not labelled** — two-cable triangulation GT must be built (unsolved association step).
3. **Filename encodings collide** — annotation `filename` uses colons, bucket objects dashes, CSV basenames underscores.
4. **Segment overlap** — files step 50 s but segments are 70 s → ~20 s overlap → ~7.5 % duplicate picks near boundaries. Dedup first.
5. **`call_id`/`segment_id` are per-file** — never key tracks globally across files.
6. **Two-cable leakage** — the same whale appears on both cables; split train/val by cable and/or batch, never by row.
7. **Instrument artifacts** — the <12.5 km void and South 28–33 km dead band are not biology.
8. **Extreme pixel imbalance** — ~2.5 % positive over 185 M/band; ~4 TB/day/band. I/O + focal/ridge framing dominate.